# Validation of model_predict_qubit_TransmonCross_cap_matrix wtih Ansys Q3D

In [1]:
from squadds import SQuADDS_DB
import pandas as pd
from squadds import Analyzer
import matplotlib.pyplot as plt
from squadds import AnsysSimulator
import numpy as np

We don't actually care about using SQuADDS to get a device at the moment, we just use the code block below to get a "template" of the SQUaDDS style dataframe.

In [2]:
''' grab SQuADDS entry as a template for ML predicted designs ''' 
db = SQuADDS_DB()
db.select_system("qubit")
db.select_qubit("TransmonCross")
df = db.create_system_df()

analyzer = Analyzer(db)

# we are not actually looking for these Hamiltonian parameters... 
target_params={"qubit_frequency_GHz": 4, "anharmonicity_MHz": -200}

pred_df = analyzer.find_closest(target_params=target_params,
                                       num_top=1,
                                       metric="Euclidean",
                                       display=True)

''' read in ML results ''' 
ML_results = pd.read_csv("predictions_and_errors_unscaled_one_hot.csv") # real in ML test results

## Sweep through ML design results, simulate each design with Anysys Q3D, and save sim results
There are only three multi-valued parameters in the SQuADDS qubit-TransmonCross-cap_matrix dataset:
* connection_pads.readout.claw_length
* connection_pads.readout.ground_spacing
* cross_length

The model only predicts these three design parameters. We take a SQuADDS database entry and substitute in the predicted parameters from the model.

In [3]:
# dictionary to save results in 
results = pd.DataFrame({"Sample":[],
                   "ref_design":[],
                   "pred_design":[],
                   "ref_H_params":[],
                   "pred_H_params":[]})

um = 10**6 ## ML model is trained in SI units (m), convert back to µm  
samples_to_test = np.arange(0,10,1)

In [4]:
for sample in samples_to_test:

    ''' current testing sample '''
    this_device = ML_results[ML_results.sample_idx == sample]

    ''' get ML predicted design parameters '''
    # reference/truth device parameters
    ref_claw_length = str(this_device.ref_unscaled.iloc[0] * um)+'um' # grab device params, convert back to microns, and add unit labels
    ref_ground_spacing = str(this_device.ref_unscaled.iloc[1] * um)+'um'
    ref_cross_length = str(this_device.ref_unscaled.iloc[2] * um)+'um'
    
    # reference/truth Hamiltonian parameters
    ref_Hamiltonian_params = {"qubit_frequency_GHz":this_device.qubit_frequency_GHz.iloc[0],"anharmonicity_MHz":this_device.anharmonicity_MHz.iloc[0]}
    
    # predicted device parameters
    pred_claw_length = str(this_device.pred_unscaled.iloc[0] * um)+'um'
    pred_ground_spacing = str(this_device.pred_unscaled.iloc[1] * um)+'um'
    pred_cross_length = str(this_device.pred_unscaled.iloc[2] * um)+'um'

    ''' create our predicted design option for Qiskit Metal '''
    pred_df.design_options.iloc[0]["connection_pads"]["readout"]["claw_length"] = pred_claw_length
    pred_df.design_options.iloc[0]["connection_pads"]["readout"]["ground_spacing"] = pred_ground_spacing
    pred_df.design_options.iloc[0]["cross_length"] = pred_cross_length
    pred_device = pred_df.iloc[0]

    ''' simulate predicted design '''
    pred_ansys_simulator = AnsysSimulator(analyzer, pred_device)
    pred_ansys_results = pred_ansys_simulator.simulate(pred_device)
    pred_Hamiltonian_params = pred_ansys_simulator.get_xmon_info(pred_ansys_results)

    ''' save results '''
    ref_design = {"ref_claw_length":ref_claw_length,"ref_ground_spacing":ref_ground_spacing,"ref_cross_length":ref_cross_length}
    pred_design = {"pred_claw_length":pred_claw_length,"pred_ground_spacing":pred_ground_spacing,"pred_cross_length":pred_cross_length}
    
    results.loc[len(results)] = [sample,ref_design,pred_design,ref_Hamiltonian_params,pred_Hamiltonian_params]

selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 10:28AM [connect_project]: Connecting to Ansys Desktop API...
INFO 10:28AM [load_ansys_project]: 	Opened Ansys App
INFO 10:28AM [load_ansys_project]: 	Opened Ansys Desktop v2023.2.0
INFO 10:28AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/firas/Documents/Ansoft/
	Project:   Project7
INFO 10:28AM [connect_design]: No active design found (or error getting active design).
INFO 10:28AM [connect]: 	 Connected to project "Project7". No design detected
INFO 10:28AM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d [Solution type: Q3D]
WARNING 10:28AM [connect_setup]: 	No design setup detected.
WARNING 10:28AM [connect_setup]: 	Creating Q3D default setup.
INFO 10:28AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:28AM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:28AM [analyze]: Analyzing setup sweep_setup
INFO 10:34AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppD

qubit anharmonicity = -139 MHz 
qubit frequency = 3.961 GHz
selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 10:34AM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d1 [Solution type: Q3D]
WARNING 10:34AM [connect_setup]: 	No design setup detected.
WARNING 10:34AM [connect_setup]: 	Creating Q3D default setup.
INFO 10:34AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:34AM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 10:34AM [analyze]: Analyzing setup sweep_setup
INFO 10:41AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmprs3denpb.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 10:41AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmp7_f9shxj.txt, C, , sweep_setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 10:41AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmppfynwnmx.txt, C, , sweep_setup:AdaptiveP

qubit anharmonicity = -127 MHz 
qubit frequency = 3.799 GHz
selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


...

qubit anharmonicity = -109 MHz 
qubit frequency = 3.545 GHz
selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 11:22AM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d8 [Solution type: Q3D]
WARNING 11:22AM [connect_setup]: 	No design setup detected.
WARNING 11:22AM [connect_setup]: 	Creating Q3D default setup.
INFO 11:22AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:22AM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:22AM [analyze]: Analyzing setup sweep_setup
INFO 11:29AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpo3zhg8lx.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 11:29AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpmkjo7lba.txt, C, , sweep_setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 11:29AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmptuievw0d.txt, C, , sweep_setup:AdaptiveP

qubit anharmonicity = -107 MHz 
qubit frequency = 3.509 GHz
selected system: qubit
the parameters ['run'] are unsupported, so they have been ignored


INFO 11:29AM [connect_design]: 	Opened active design
	Design:    LOMv2.0_q3d9 [Solution type: Q3D]
WARNING 11:29AM [connect_setup]: 	No design setup detected.
WARNING 11:29AM [connect_setup]: 	Creating Q3D default setup.
INFO 11:29AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:29AM [get_setup]: 	Opened setup `sweep_setup`  (<class 'pyEPR.ansys.AnsysQ3DSetup'>)
INFO 11:29AM [analyze]: Analyzing setup sweep_setup
INFO 11:36AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmpv3t4r102.txt, C, , sweep_setup:LastAdaptive, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 11:36AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmp59ri65kj.txt, C, , sweep_setup:AdaptivePass, "Original", "ohm", "nH", "fF", "mSie", 5000000000, Maxwell, 1, False
INFO 11:36AM [get_matrix]: Exporting matrix data to (C:\Users\firas\AppData\Local\Temp\34\tmp87mziz48.txt, C, , sweep_setup:AdaptiveP

qubit anharmonicity = -145 MHz 
qubit frequency = 4.041 GHz


## Save the data

In [8]:
results.to_csv("transmonCapMat_simulation_results.csv",index=False) # save data for later analysis with validation_01_data_analysis.ipynb